# Gradio Day — Reference Notebook

Build LLM-backed UIs with Gradio: from a one-line Interface, through streaming and multi-model selection, to a company brochure generator.

**How to use this notebook**
- Run cells top-to-bottom the first time (later cells depend on earlier ones).
- Each Gradio `launch()` starts a local server; stop old ones or ignore unused ports.
- Secrets live in a `.env` file (never hard-code API keys).

In [1]:
# Core libraries for this lab:
# - os / dotenv: load API keys from a local .env file
# - OpenAI: official SDK — also used as a thin client for Anthropic & Gemini
#   via their OpenAI-compatible endpoints (same chat.completions API shape)
# - gradio: turns Python functions into interactive web UIs with almost no HTML/JS
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

# Part 1 — Establish LLM clients

Load keys, create OpenAI-compatible clients for OpenAI / Anthropic / Gemini, then define a simple non-streaming chat helper.

In [2]:
# Load variables from .env into the process environment.
# override=True means .env values win over any already-set shell env vars.
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")

# Print only a short prefix so you can confirm the key loaded without leaking secrets.
# If a print says "not set", check your .env path / variable names before calling any API.
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:5]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AQ.Ab 


In [3]:
# Three clients, one SDK pattern (OpenAI chat.completions).
#
# openai  → default OpenAI API (reads OPENAI_API_KEY from env automatically)
# anthropic / gemini → same OpenAI() class pointed at provider-specific base_url
#   so you can call .chat.completions.create(...) the same way for all three.
#
# Tip: if you skip Anthropic or Google, comment out that client (and later stream_* helpers).
openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [4]:
# Non-streaming chat helper used by early Gradio demos.
#
# Chat APIs expect a list of role/content dicts:
#   system → behavior instructions (persona, format, constraints)
#   user   → the current prompt
#
# max_tokens caps reply length (cheap/fast for demos; raise for real answers).
# response.choices[0].message.content is the assistant text for a normal completion.
system_message = "You are a helpful assistant"

def message_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt},
    ]
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        max_tokens=100,
    )
    return response.choices[0].message.content


In [5]:
# Smoke-test the OpenAI client before wiring Gradio.
# Note: many models do not know the true "today"; they may invent a date.
message_gpt("What is today's date?")

"Today's date is June 7, 2024."

# Part 2 — Gradio user interfaces

Pattern: write a plain Python function → wrap it with `gr.Interface` → `launch()`.
Gradio inspects inputs/outputs and builds the web UI for you.

In [6]:
# Tiny demo function with no LLM — proves Gradio wiring before adding API calls.
# print() shows up in the notebook/server logs; return value becomes the UI output.
def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()

In [7]:
# Call the function directly (no UI) to confirm behavior.
shout("hello")

Shout has beed called with input hello


'HELLO'

In [8]:
# Minimal Gradio app:
#   fn      → Python function to run on submit
#   inputs  → widget type for arguments (string shorthand works for simple cases)
#   outputs → widget type for the return value
#   flagging_mode="never" → hide Gradio's "Flag" button (useful in demos)
#
# launch() starts a local web server and prints a URL (often http://127.0.0.1:78xx).
gr.Interface(
    fn=shout,
    inputs="textbox",
    outputs="textbox",
    flagging_mode="never",
).launch()


* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


In [9]:
# share=True asks Gradio to create a temporary public URL (*.gradio.live).
# Useful for demos on another device; links expire (often ~1 week).
# Prefer auth / private deploy for anything sensitive.
gr.Interface(
    fn=shout,
    inputs="textbox",
    outputs="textbox",
    flagging_mode="never",
).launch(share=True)


* Running on local URL:  http://127.0.0.1:7873
* Running on public URL: https://fc49c41520501f73ac.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
# inbrowser=True opens the UI in your default browser when the server is ready.
gr.Interface(
    fn=shout,
    inputs="textbox",
    outputs="textbox",
    flagging_mode="never",
).launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


## 2.1 Adding authentication

Pass `auth=(username, password)` to `launch()` for a basic login gate on the Gradio UI.
This is convenience auth for demos — not a production security model.


In [11]:
# Basic username/password prompt before the app loads.
# Replace with real credentials (or better auth) if you ever share beyond local use.
gr.Interface(
    fn=shout,
    inputs="textbox",
    outputs="textbox",
    flagging_mode="never",
).launch(
    inbrowser=True,
    auth=("michael", "1234556789"),
)


* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


## 2.2 Dark mode

Gradio honors a `__theme=dark` URL query param. Inject a tiny JS snippet via `js=` so the page reloads once into dark mode.


In [12]:
# Custom JS run in the browser when the page loads.
# If the URL is not already dark-themed, set __theme=dark and reload.
force_dark_mode = """
function refresh() {
    const url = new URL(window.location);

    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""

gr.Interface(
    fn=shout,
    inputs="textbox",
    outputs="textbox",
    flagging_mode="never",
    js=force_dark_mode,  # inject client-side behavior
).launch()


* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


## 2.3 Titles, labels, and examples

Build richer UIs by constructing `gr.Textbox` (and similar) widgets yourself, then pass them into `Interface`.
`examples=` adds one-click sample inputs under the form.


In [13]:
# Explicit widgets give you labels, helper text (info), and sizing (lines).
message_input = gr.Textbox(
    label="Your message:",
    info="Enter a message to be shouted",
    lines=7,
)
message_output = gr.Textbox(label="Response", lines=8)

# Store the Interface in `view` so you can call view.launch() separately if needed.
view = gr.Interface(
    fn=shout,
    title="Shout",                 # app title shown at the top
    inputs=[message_input],        # list matches fn argument order
    outputs=[message_output],
    examples=["hello", "howdy"],   # clickable sample prompts
    flagging_mode="never",
)

view.launch()


* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


In [14]:
# Same UI pattern, but fn=message_gpt so Gradio calls the LLM instead of shout().
# Gradio passes the textbox value as the first (and only) argument to message_gpt.
message_input = gr.Textbox(
    label="Your message:",
    info="Enter a message for GPT-4.1-mini",
    lines=7,
)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=message_gpt,
    title="GPT",
    inputs=[message_input],
    outputs=[message_output],
    examples=["hello", "howdy"],
    flagging_mode="never",
)

view.launch()


* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.


In [15]:
# Repeat of the GPT Interface (kept as a practice / checkpoint cell).
# Tip: if you already launched an identical UI above, you can skip re-running this.
message_input = gr.Textbox(
    label="Your message:",
    info="Enter a message for GPT-4.1-mini",
    lines=7,
)
message_output = gr.Textbox(label="Response", lines=8)

view = gr.Interface(
    fn=message_gpt,
    title="GPT",
    inputs=[message_input],
    outputs=[message_output],
    examples=["hello", "howdy"],
    flagging_mode="never",
)

view.launch()


* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


## 2.4 Markdown output

Swap `gr.Textbox` for `gr.Markdown` so headings, lists, and bold render instead of showing raw `**markdown**` characters.
Ask the model (via system_message) to respond in markdown for best results.


In [16]:
# Update system instructions so answers are markdown-friendly.
# "without code blocks" avoids fenced ``` blocks that can look awkward in some UIs.
system_message = (
    "You are a helpful assistant that responds in markdown without code blocks"
)

message_input = gr.Textbox(
    label="Your message:",
    info="Enter a message for GPT-4.1-mini",
    lines=7,
)
# Markdown output renders formatting; Textbox would show raw markdown syntax.
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=message_gpt,
    title="GPT",
    inputs=[message_input],
    outputs=[message_output],
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
    ],
    flagging_mode="never",
)
view.launch()


* Running on local URL:  http://127.0.0.1:7880
* To create a public link, set `share=True` in `launch()`.


## 2.5 Streaming results

With `stream=True`, the API yields token chunks as they arrive.
Make your function a **generator** (`yield` partial text) so Gradio updates the UI live instead of waiting for the full reply.


In [17]:
# Streaming GPT helper.
# Key idea: accumulate text in `result`, yield after each chunk so Gradio refreshes.
#
# chunk.choices[0].delta.content is often None on the first/last events — use `or ""`.
# Gradio detects generators automatically when fn yields strings.
def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt},
    ]
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        stream=True,  # enable token-by-token delivery
    )

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result  # partial answer so far


In [18]:
# Identical UI to the markdown demo, but fn=stream_gpt → live typing effect.
message_input = gr.Textbox(
    label="Your message",
    info="Enter a message for GPT-4.1-mini",
    lines=7,
)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_gpt,
    title="GPT",
    inputs=[message_input],
    outputs=[message_output],
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
    ],
    flagging_mode="never",
)

view.launch()


* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.


In [19]:
# Streaming Claude via Anthropic's OpenAI-compatible endpoint (same API shape as GPT).
#
# Model IDs are exact strings — e.g. claude-sonnet-4-5-20250929
# (not claude-4-5-sonnet-...). A wrong ID returns openai.NotFoundError / 404.
#
# Message roles mirror the GPT helper: system + user (not assistant).
def stream_claude(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt},
    ]
    stream = anthropic.chat.completions.create(
        model="claude-sonnet-4-5-20250929",
        messages=messages,
        stream=True,
    )

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result


In [20]:
# Same streaming Markdown UI pattern, pointed at Claude.
message_input = gr.Textbox(
    label="Your message:",
    info="Enter a message for Claude 4.5 Sonnet",
    lines=7,
)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_claude,
    title="Claude",
    inputs=[message_input],
    outputs=[message_output],
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
    ],
    flagging_mode="never",
)
view.launch()


* Running on local URL:  http://127.0.0.1:7882
* To create a public link, set `share=True` in `launch()`.


## 2.6 Using multiple LLMs

One Interface, two models: add a Dropdown input and route inside a dispatcher that `yield from` the chosen stream helper.


In [21]:
# Dispatcher: Gradio will call stream_model(prompt, model) because we pass two inputs.
# `yield from result` forwards every partial string from stream_gpt / stream_claude
# so the UI still streams instead of buffering the whole answer.
def stream_model(prompt, model):
    if model == "GPT":
        result = stream_gpt(prompt)
    elif model == "Claude":
        result = stream_claude(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result


In [22]:
# Multi-input Interface:
#   inputs[0] → prompt textbox  → stream_model's `prompt`
#   inputs[1] → model dropdown  → stream_model's `model`
#
# Examples must be lists matching the input order: [prompt, model].
message_input = gr.Textbox(
    label="Your message:",
    info="Enter a message for the LLM",
    lines=7,
)
model_selector = gr.Dropdown(
    ["GPT", "Claude"],
    label="Select model",
    value="GPT",  # default selection
)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="LLMs",
    inputs=[message_input, model_selector],
    outputs=[message_output],
    examples=[
        ["Explain the Transformer architecture to a layperson", "GPT"],
        ["Explain the Transformer architecture to an aspiring AI engineer", "Claude"],
    ],
    flagging_mode="never",
)

view.launch()


* Running on local URL:  http://127.0.0.1:7883
* To create a public link, set `share=True` in `launch()`.


# 3. Building a company brochure generator

End-to-end mini-app:
1. Scrape landing-page text (`fetch_website_contents`)
2. Build a prompt with company name + page content
3. Stream a markdown brochure with GPT or Claude via Gradio


In [ ]:
# This notebook lives under my-work/Week 2/, but the course helper is in llm_engineering/week2/scraper.py.
# Add that folder to sys.path so `from scraper import ...` finds the course module —
# not an unrelated PyPI package named "scraper" that may be installed in the venv.
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../../week2").resolve()))
from scraper import fetch_website_contents


In [25]:
# Brochure-specific system prompt: role, audience, and output format.
# Replaces the earlier generic assistant message used by stream_gpt / stream_claude.
system_message = """
You are an assistant that analyzes the contents of a company
website landing page and creates a short brochure about the
company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""


In [26]:
# Brochure pipeline:
# 1) yield "" immediately so Gradio clears / shows an empty response while scraping
# 2) fetch page text and append it to the user prompt
# 3) stream from the selected model (reuses stream_gpt / stream_claude)
#
# Note: stream_gpt/claude read the global `system_message` defined above —
# re-run this cell (and those helpers) after changing the system prompt.
def stream_brochure(company_name, url, model):
    yield ""  # clear previous output while we scrape / call the LLM
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)

    if model == "GPT":
        result = stream_gpt(prompt)
    elif model == "Claude":
        result = stream_claude(prompt)
    else:
        raise ValueError("Unknown Model")

    yield from result


In [29]:
# Three inputs → stream_brochure(company_name, url, model)
# Examples include a known-good URL so you can demo without hunting for sites.
name_input = gr.Textbox(label="Company Name")
url_input = gr.Textbox(label="Company URL")
model_selector = gr.Dropdown(
    ["GPT", "Claude"],
    label="Select model",
    value="GPT",
)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator",
    inputs=[name_input, url_input, model_selector],
    outputs=[message_output],
    examples=[
        ["Hugging Face", "https://huggingface.co", "GPT"],
        ["Edward Donner", "https://edwarddonner.com", "Claude"],
    ],
    flagging_mode="never",
)

view.launch()


* Running on local URL:  http://127.0.0.1:7884
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/michaelperrine/Documents/Donner/ai-engineer-core-donner/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 849, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/michaelperrine/Documents/Donner/ai-engineer-core-donner/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/michaelperrine/Documents/Donner/ai-engineer-core-donner/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/michaelperrine/Documents/Donner/ai-engineer-core-donner/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1635, in call_function
    